In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd

ROOT = Path("/kaggle/input")

def find_csv(filename):
    paths = list(ROOT.rglob(filename))
    assert len(paths) == 1, paths
    return paths[0]

binary_path = find_csv(
    "diabetes_binary_health_indicators_BRFSS2015.csv"
)

three_path = find_csv(
    "diabetes_012_health_indicators_BRFSS2015.csv"
)

def sha256_file(path):
    return hashlib.sha256(
        path.read_bytes()
    ).hexdigest()

binary = pd.read_csv(binary_path)
three = pd.read_csv(three_path)

print("=== DATASET IDENTITY ===")
print("Binary:", binary.shape)
print("3-class:", three.shape)

print("Binary SHA-256:")
print(sha256_file(binary_path))

print("3-class SHA-256:")
print(sha256_file(three_path))

print("\n=== FEATURE ALIGNMENT ===")

X_binary = binary.drop(columns="Diabetes_binary")
X_three = three.drop(columns="Diabetes_012")

assert X_binary.columns.tolist() == X_three.columns.tolist()
assert X_binary.shape == X_three.shape

same_X = np.array_equal(
    X_binary.to_numpy(),
    X_three.to_numpy()
)

print("Identical feature vectors and order:", same_X)

assert same_X, "Cannot compare labels row by row"

print("\n=== LABEL CROSS-TABULATION ===")

mapping = pd.crosstab(
    three["Diabetes_012"],
    binary["Diabetes_binary"]
)

print(mapping)

print("\n=== LABEL HYPOTHESES ===")

combined = (three["Diabetes_012"] >= 1).astype(int)
diabetes_only = (three["Diabetes_012"] == 2).astype(int)

y = binary["Diabetes_binary"]

print(
    "Binary = prediabetes + diabetes:",
    np.array_equal(y, combined)
)

print(
    "Binary = diabetes only:",
    np.array_equal(y, diabetes_only)
)

print("\nS9 LABEL AUDIT: PASS")

=== DATASET IDENTITY ===
Binary: (253680, 22)
3-class: (253680, 22)
Binary SHA-256:
19f367e3e3350768f0c144c5d73ee5b355f67a57eaaa86ca7bd8aec594d8b1d0
3-class SHA-256:
a971809d0786d2d7d7f6070f83dfe8af9d860ea33a79e0e2701c8a49f78f9861

=== FEATURE ALIGNMENT ===
Identical feature vectors and order: True

=== LABEL CROSS-TABULATION ===
Diabetes_binary     0.0    1.0
Diabetes_012                  
0.0              213703      0
1.0                4631      0
2.0                   0  35346

=== LABEL HYPOTHESES ===
Binary = prediabetes + diabetes: False
Binary = diabetes only: True

S9 LABEL AUDIT: PASS


In [2]:

# ==========================================
# S9 - CELL 2: LABEL AUDIT EVIDENCE
# ==========================================

import json
import zipfile
import platform
from datetime import datetime, timezone

AUDIT_DIR = Path("/kaggle/working/S9_Label_Audit")
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Verify dataset identity

EXPECTED_BINARY_HASH = (
    "19f367e3e3350768f0c144c5d73ee5b355f67a57eaaa86ca7bd8aec594d8b1d0"
)

EXPECTED_THREE_HASH = (
    "a971809d0786d2d7d7f6070f83dfe8af9d860ea33a79e0e2701c8a49f78f9861"
)

assert sha256_file(binary_path) == EXPECTED_BINARY_HASH
assert sha256_file(three_path) == EXPECTED_THREE_HASH

assert binary.shape == (253680, 22)
assert three.shape == (253680, 22)

assert X_binary.columns.tolist() == X_three.columns.tolist()

assert np.array_equal(
    X_binary.to_numpy(),
    X_three.to_numpy()
)

print("[PASS] Dataset identity and alignment")


# 2. Verify label domains

assert set(binary["Diabetes_binary"].unique()) == {0, 1}
assert set(three["Diabetes_012"].unique()) == {0, 1, 2}

mapping = pd.crosstab(
    three["Diabetes_012"],
    binary["Diabetes_binary"]
)

expected_mapping = np.array([
    [213703, 0],
    [4631, 0],
    [0, 35346]
])

np.testing.assert_array_equal(
    mapping.to_numpy(),
    expected_mapping
)

assert mapping.to_numpy().sum() == 253680

print("[PASS] Label cross-tabulation")


# 3. Verify both label hypotheses

y_binary = binary["Diabetes_binary"].to_numpy()

y_combined = (
    three["Diabetes_012"] >= 1
).astype(int).to_numpy()

y_diabetes = (
    three["Diabetes_012"] == 2
).astype(int).to_numpy()

combined_match = np.array_equal(
    y_binary, y_combined
)

diabetes_match = np.array_equal(
    y_binary, y_diabetes
)

assert not combined_match
assert diabetes_match

combined_mismatch = int(
    np.sum(y_binary != y_combined)
)

diabetes_mismatch = int(
    np.sum(y_binary != y_diabetes)
)

assert combined_mismatch == 4631
assert diabetes_mismatch == 0

print("[PASS] Label mapping hypotheses")


# 4. Save cross-tabulation

mapping.to_csv(
    AUDIT_DIR / "label_cross_tabulation.csv"
)


# 5. Save hypothesis results

hypotheses = pd.DataFrame([
    {
        "hypothesis": "prediabetes_plus_diabetes",
        "exact_match": bool(combined_match),
        "mismatched_rows": combined_mismatch
    },
    {
        "hypothesis": "diabetes_only",
        "exact_match": bool(diabetes_match),
        "mismatched_rows": diabetes_mismatch
    }
])

hypotheses.to_csv(
    AUDIT_DIR / "label_hypothesis_results.csv",
    index=False
)


# 6. Save audit report

audit_report = {
    "task": "S9_label_semantics_audit",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "status": "PASS",

    "binary_filename": binary_path.name,
    "three_class_filename": three_path.name,

    "binary_sha256": sha256_file(binary_path),
    "three_class_sha256": sha256_file(three_path),

    "n_records": len(binary),
    "n_features": X_binary.shape[1],

    "feature_names": X_binary.columns.tolist(),

    "identical_feature_values_and_order": True,

    "label_mapping": {
        "three_class_0_to_binary_0": 213703,
        "three_class_1_to_binary_0": 4631,
        "three_class_2_to_binary_1": 35346
    },

    "binary_negative": int((y_binary == 0).sum()),
    "binary_positive": int((y_binary == 1).sum()),

    "combined_label_hypothesis": {
        "match": bool(combined_match),
        "mismatches": combined_mismatch
    },

    "diabetes_only_hypothesis": {
        "match": bool(diabetes_match),
        "mismatches": diabetes_mismatch
    },

    "conclusion":
        "Binary label equals indicator of Diabetes_012 == 2 "
        "in the audited CSV files.",

    "scope": "Full dataset label semantics audit",

    "model_training_performed": False,
    "performance_evaluation_performed": False,
    "model_selection_performed": False,

    "original_labels_modified": False,
    "split_indices_modified": False,

    "team_approval_status": "PENDING",

    "limitations": [
        "The audit verifies the empirical mapping "
        "between the two CSV label columns.",

        "The audit does not independently establish "
        "clinical diagnosis or participant identity.",

        "The full dataset was accessed for label auditing, "
        "not for model performance evaluation."
    ],

    "environment": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__
    }
}

with open(
    AUDIT_DIR / "S9_LABEL_AUDIT_REPORT.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        audit_report,
        f,
        indent=2,
        ensure_ascii=False
    )


# 7. Create README

readme = """# S9 - Label Semantics Audit

Dataset: BRFSS 2015 Diabetes Health Indicators.

This audit compares the binary and 3-class CSV files.

Results:
- 253,680 records.
- 21 identical feature columns in identical row order.
- Binary label equals (Diabetes_012 == 2).
- All 4,631 records with Diabetes_012 == 1
  map to binary label 0.

No model training or performance evaluation
was performed.

Original labels and split indices remain unchanged.

Team approval of the revised target interpretation
is pending.

This package contains aggregate audit evidence only.
"""

(AUDIT_DIR / "README_S9_Label_Audit.md").write_text(
    readme,
    encoding="utf-8"
)


# 8. Generate SHA-256 manifest

evidence_names = [
    "label_cross_tabulation.csv",
    "label_hypothesis_results.csv",
    "S9_LABEL_AUDIT_REPORT.json",
    "README_S9_Label_Audit.md"
]

manifest = {}

for name in evidence_names:

    path = AUDIT_DIR / name

    manifest[name] = {
        "sha256": sha256_file(path),
        "bytes": path.stat().st_size
    }

with open(
    AUDIT_DIR / "sha256_manifest.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(manifest, f, indent=2)

package_names = evidence_names + [
    "sha256_manifest.json"
]


# 9. Create ZIP evidence

ZIP_PATH = Path(
    "/kaggle/working/S9_Label_Audit_Evidence.zip"
)

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:

    for name in package_names:

        z.write(
            AUDIT_DIR / name,
            arcname=f"S9_Label_Audit/{name}"
        )


# 10. Verify archive

with zipfile.ZipFile(ZIP_PATH, "r") as z:

    assert z.testzip() is None

    assert set(z.namelist()) == {
        f"S9_Label_Audit/{name}"
        for name in package_names
    }

    for name, info in manifest.items():

        content = z.read(
            f"S9_Label_Audit/{name}"
        )

        assert hashlib.sha256(
            content
        ).hexdigest() == info["sha256"]

        assert len(content) == info["bytes"]

print("[PASS] ZIP and manifest integrity")


# 11. Final summary

print("\n=== S9 LABEL AUDIT SUMMARY ===")

print(mapping)

print("\n", hypotheses.to_string(index=False))

print("\nEvidence files:", len(package_names))
print("Manifest matches:", len(manifest), "/ 4")
print("ZIP:", ZIP_PATH)
print("ZIP SHA-256:", sha256_file(ZIP_PATH))

print("\nModel training: NO")
print("Performance evaluation: NO")
print("Original labels modified: NO")
print("Team approval: PENDING")

print("\nS9 LABEL AUDIT EVIDENCE: PASS")


[PASS] Dataset identity and alignment
[PASS] Label cross-tabulation
[PASS] Label mapping hypotheses
[PASS] ZIP and manifest integrity

=== S9 LABEL AUDIT SUMMARY ===
Diabetes_binary     0.0    1.0
Diabetes_012                  
0.0              213703      0
1.0                4631      0
2.0                   0  35346

                hypothesis  exact_match  mismatched_rows
prediabetes_plus_diabetes        False             4631
            diabetes_only         True                0

Evidence files: 5
Manifest matches: 4 / 4
ZIP: /kaggle/working/S9_Label_Audit_Evidence.zip
ZIP SHA-256: 9d26662a35e66c52ed7eb209d7f74075871a0e272bff259d9d0832e8226ce3d5

Model training: NO
Performance evaluation: NO
Original labels modified: NO
Team approval: PENDING

S9 LABEL AUDIT EVIDENCE: PASS
